# NCBI Entrez Tools

![NCBI Entrez Tools](https://proto-bio.github.io/proto-assets/images/tool/ncbi/hero.png)

This notebook demonstrates the three NCBI E-utility wrappers provided by proto_tools. The `run_ncbi_esearch` function finds database record IDs matching a query term, `run_ncbi_esummary` retrieves structured metadata for a given record, and `run_ncbi_efetch` downloads full sequences in FASTA format. Together, these three functions support a common bioinformatics workflow: search for records of interest, inspect their metadata, and then fetch the actual sequence data for downstream analysis. A fourth function, `run_ncbi_reference_bundle`, downloads a whole version-pinned genome assembly from NCBI Datasets.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("ncbi")
display_overview("ncbi")
display_docs_section("ncbi", "Background")

# NCBI Entrez

[NCBI Entrez](https://www.ncbi.nlm.nih.gov/search/) is the [National Center for Biotechnology Information](https://www.ncbi.nlm.nih.gov/)'s search and retrieval system over its biological sequence databases, accessed through the Entrez Programming Utilities (E-utilities). This toolkit wraps three E-utilities endpoints: `ncbi-esearch` (query term to matching record UIDs), `ncbi-esummary` (document-summary metadata for a UID or accession), and `ncbi-efetch` (full sequence records as parsed FASTA for the protein and nucleotide databases). A fourth tool, `ncbi-reference-bundle`, downloads whole genome assemblies from the separate [NCBI Datasets](https://www.ncbi.nlm.nih.gov/datasets/) service.

[NCBI Entrez](https://www.ncbi.nlm.nih.gov/search/) and its underlying sequence archive are described in the GenBank report ([Sayers et al., 2022](https://doi.org/10.1093/nar/gkab1135)), published in *Nucleic Acids Research*. The Entrez system and the [E-utilities](https://www.ncbi.nlm.nih.gov/books/NBK25501/) are operated by the [National Center for Biotechnology Information](https://www.ncbi.nlm.nih.gov/) (NCBI), part of the [U.S. National Library of Medicine](https://www.nlm.nih.gov/) (NLM). Entrez unifies search and retrieval across more than forty interconnected databases, including the protein, nucleotide, and gene databases used here, with records drawn from sources such as [RefSeq](https://www.ncbi.nlm.nih.gov/refseq/) and [GenBank](https://www.ncbi.nlm.nih.gov/genbank/).

Internally, each tool issues an HTTP GET to the E-utilities base endpoint `https://eutils.ncbi.nlm.nih.gov/entrez/eutils`. `ncbi-esearch` calls `esearch.fcgi` and returns the JSON `idlist`. `ncbi-esummary` calls `esummary.fcgi` and returns the JSON result map. `ncbi-efetch` calls `efetch.fcgi` with a FASTA `rettype` and parses the response into records. Every request carries a fixed `tool=` identifier. The `email=` and `api_key=` parameters are sent only when `ncbi_email` and `ncbi_api_key` are configured. The request URL surfaced on outputs is sanitized so the API key and email are stripped before it is returned. Records and their provenance come directly from NCBI's live E-utilities, so results reflect the database state at query time rather than a fixed release snapshot. `ncbi-reference-bundle` instead calls the NCBI Datasets v2 API ([O'Leary et al., 2024](https://doi.org/10.1038/s41597-024-03571-y)), whose versioned assembly accessions name fixed sequence content.

## Available tools

In [2]:
display_available_tools("ncbi")

- **`run_ncbi_efetch()`** — Fetch FASTA records from NCBI sequence dbs (protein/nuccore) by accession or ID
- **`run_ncbi_esearch()`** — Search NCBI Entrez databases by query term to find matching IDs
- **`run_ncbi_esummary()`** — Retrieve record summary metadata from NCBI Entrez by ID
- **`run_ncbi_reference_bundle()`** — Download a version-pinned NCBI Datasets genome reference bundle

### `run_ncbi_esearch`

The first step in most NCBI workflows is finding the database IDs for records of interest. `run_ncbi_esearch` accepts a free-text query using NCBI's Entrez search syntax, which supports field-qualified terms like gene names and organism filters. It returns a list of matching record IDs from the protein, nuccore, or gene database, which can then be passed directly to `run_ncbi_esummary` or `run_ncbi_efetch` for further retrieval.

In [3]:
from proto_tools import (
    NCBIEsearchInput,
    NCBIEsearchConfig,
    run_ncbi_esearch,
)

In [4]:
# Display docs
display_api_reference("ncbi", "input", "run_ncbi_esearch")

# Search for lac repressor protein IDs in Escherichia coli
inputs = NCBIEsearchInput(
    db="protein",
    search_term='"lacI"[Gene] AND "Escherichia coli"[Organism]',
    max_results=5,
)

**Input** — `NCBIEsearchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>db</code> | <code>Literal['protein', 'nuccore', 'nucleotide', 'gene', 'pubmed', 'pmc', 'taxonomy', 'structure', 'snp', 'clinvar', 'omim', 'biosample', 'bioproject', 'sra', 'assembly', 'ipg', 'mesh', 'genome', 'dbvar', 'gds', 'geoprofiles', 'medgen', 'proteinclusters', 'protfam', 'pccompound', 'pcsubstance', 'pcassay']</code> | required | NCBI database to query (e.g. 'protein', 'nuccore', 'gene', 'pubmed', 'taxonomy') |
| <code>search_term</code> | <code>str</code> | required | NCBI search query (e.g. 'lacI[Gene] AND Escherichia coli[Organism]') |
| <code>max_results</code> | <code>int</code> | <code>20</code> | Max IDs to return (NCBI retmax; default 20) |
| <code>retstart</code> | <code>int</code> | <code>0</code> | 0-indexed offset of the first hit (NCBI retstart). For pagination. |
| <code>sort</code> | <code>str &#124; None</code> | <code>None</code> | Db-specific sort key (e.g. 'relevance' or 'pub_date' on pubmed) |
| <code>field</code> | <code>str &#124; None</code> | <code>None</code> | Restrict search term to a single field (e.g. 'title' or 'author' on pubmed) |
| <code>datetype</code> | <code>Optional[Literal['mdat', 'pdat', 'edat']]</code> | <code>None</code> | Date axis for mindate/maxdate/reldate (one of mdat, pdat, edat) |
| <code>mindate</code> | <code>str &#124; None</code> | <code>None</code> | Lower date bound (YYYY/MM/DD, YYYY/MM, or YYYY); requires datetype |
| <code>maxdate</code> | <code>str &#124; None</code> | <code>None</code> | Upper date bound (YYYY/MM/DD, YYYY/MM, or YYYY); requires datetype |
| <code>reldate</code> | <code>int &#124; None</code> | <code>None</code> | Records dated within the last N days; requires datetype |

In [5]:
# Display config docs
display_api_reference("ncbi", "config", "run_ncbi_esearch")

# Each NCBI function has its own config class | see docs above for all fields
config = NCBIEsearchConfig()

**Config** — `NCBIEsearchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>ncbi_api_key</code> | <code>str &#124; None</code> | <code>$NCBI_API_KEY</code> | Optional NCBI API key (3 to 10 req/s). Falls back to $NCBI_API_KEY. |
| <code>ncbi_email</code> | <code>str &#124; None</code> | <code>$NCBI_EMAIL</code> | Optional contact email for NCBI. Falls back to $NCBI_EMAIL. |

In [6]:
# Run the search
search_result = run_ncbi_esearch(inputs, config)

In [7]:
# Display output docs
display_api_reference("ncbi", "output", "run_ncbi_esearch")

# Inspect the returned IDs
print(f"IDs found: {search_result.ids}")

**Output** — `NCBIEsearchOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>ids</code> | <code>list[str]</code> | <code>[]</code> | List of NCBI IDs found by the search |
| <code>source_url</code> | <code>str</code> | required | Sanitized request URL |

IDs found: ['3417938004', '3406707271', '3405952705', '3402599149', '3401481483']


### `run_ncbi_esummary`

Once you have record IDs from a search, `run_ncbi_esummary` retrieves structured metadata without downloading the full record. This is useful for inspecting gene annotations, genomic coordinates, organism information, and other catalog-level details. It accepts a database name and a single accession or NCBI ID, and returns the full summary document as a parsed dictionary.

In [8]:
from proto_tools import (
    NCBIEsummaryInput,
    NCBIEsummaryConfig,
    run_ncbi_esummary,
)

In [9]:
# Display docs
display_api_reference("ncbi", "input", "run_ncbi_esummary")

# Fetch the gene summary for human TP53 using its NCBI Gene ID
esummary_inputs = NCBIEsummaryInput(
    db="gene",
    identifier="7157",
)

**Input** — `NCBIEsummaryInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>db</code> | <code>Literal['protein', 'nuccore', 'nucleotide', 'gene', 'pubmed', 'pmc', 'taxonomy', 'structure', 'snp', 'clinvar', 'omim', 'biosample', 'bioproject', 'sra', 'assembly', 'ipg', 'mesh', 'genome', 'dbvar', 'gds', 'geoprofiles', 'medgen', 'proteinclusters', 'protfam', 'pccompound', 'pcsubstance', 'pcassay']</code> | required | NCBI database to query (e.g. 'protein', 'nuccore', 'gene', 'pubmed', 'taxonomy') |
| <code>identifier</code> | <code>str</code> | required | Accession or NCBI ID for esummary (e.g. 'NP_000537.3', '7157') |

In [10]:
# Display config docs
display_api_reference("ncbi", "config", "run_ncbi_esummary")

# Each NCBI function has its own config class | see docs above for all fields
esummary_config = NCBIEsummaryConfig()

**Config** — `NCBIEsummaryConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>ncbi_api_key</code> | <code>str &#124; None</code> | <code>$NCBI_API_KEY</code> | Optional NCBI API key (3 to 10 req/s). Falls back to $NCBI_API_KEY. |
| <code>ncbi_email</code> | <code>str &#124; None</code> | <code>$NCBI_EMAIL</code> | Optional contact email for NCBI. Falls back to $NCBI_EMAIL. |

In [11]:
# Run the summary fetch
summary_result = run_ncbi_esummary(esummary_inputs, esummary_config)

In [12]:
# Display output docs
display_api_reference("ncbi", "output", "run_ncbi_esummary")

# Inspect gene metadata for TP53
gene_id = esummary_inputs.identifier
gene_data = summary_result.summary.get(gene_id, {})
print(f"Name: {gene_data.get('name')}")
print(f"Description: {gene_data.get('description')}")
print(f"Organism: {gene_data.get('organism', {}).get('scientificname')}")

genomic_info = gene_data.get("genomicinfo", [])
if genomic_info:
    region = genomic_info[0]
    print(f"Chromosome: {region.get('chraccver')}")
    print(f"Coordinates: {region.get('chrstart')} to {region.get('chrstop')}")

**Output** — `NCBIEsummaryOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>summary</code> | <code>dict[str, Any]</code> | <code>{}</code> | Record summary data returned by esummary |
| <code>source_url</code> | <code>str</code> | required | Sanitized request URL |

Name: TP53
Description: tumor protein p53
Organism: Homo sapiens
Chromosome: NC_000017.11
Coordinates: 7687489 to 7668420


### `run_ncbi_efetch`

`run_ncbi_efetch` downloads full sequence records from NCBI in FASTA format. It is the workhorse for retrieving protein or nucleotide sequences needed for alignment, structure prediction, or other downstream analyses. In addition to full-record retrieval, it supports subsequence extraction by specifying start and stop coordinates (1-indexed, inclusive), and strand selection for nucleotide records, making it suitable for pulling specific gene loci or regulatory elements from large chromosome assemblies.

In [13]:
from proto_tools import (
    NCBIEfetchInput,
    NCBIEfetchConfig,
    run_ncbi_efetch,
)

In [14]:
# Display docs
display_api_reference("ncbi", "input", "run_ncbi_efetch")

# Fetch the TP53 protein sequence by RefSeq accession
efetch_inputs = NCBIEfetchInput(
    db="protein",
    identifier="NP_000537.3",
    return_format="fasta",
)

**Input** — `NCBIEfetchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>db</code> | <code>Literal['protein', 'nuccore', 'nucleotide']</code> | required | NCBI sequence database to query (protein, nuccore, or nucleotide) |
| <code>identifier</code> | <code>str</code> | required | Accession or NCBI ID for efetch (e.g. 'NP_000537.3', '7157') |
| <code>return_format</code> | <code>Literal['fasta', 'fasta_cds_na']</code> | <code>'fasta'</code> | NCBI rettype: 'fasta' (protein or nuccore) or 'fasta_cds_na' (CDS, nuccore-only) |
| <code>seq_start</code> | <code>int &#124; None</code> | <code>None</code> | Start position for subsequence extraction (1-indexed, inclusive) |
| <code>seq_stop</code> | <code>int &#124; None</code> | <code>None</code> | Stop position for subsequence extraction (1-indexed, inclusive) |
| <code>strand</code> | <code>Optional[Literal['+', '-']]</code> | <code>None</code> | Strand for nucleotide retrieval (nuccore/nucleotide only) |

In [15]:
# Display config docs
display_api_reference("ncbi", "config", "run_ncbi_efetch")

# Each NCBI function has its own config class | see docs above for all fields
efetch_config = NCBIEfetchConfig()

**Config** — `NCBIEfetchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>ncbi_api_key</code> | <code>str &#124; None</code> | <code>$NCBI_API_KEY</code> | Optional NCBI API key (3 to 10 req/s). Falls back to $NCBI_API_KEY. |
| <code>ncbi_email</code> | <code>str &#124; None</code> | <code>$NCBI_EMAIL</code> | Optional contact email for NCBI. Falls back to $NCBI_EMAIL. |

In [16]:
# Run the fetch
fetch_result = run_ncbi_efetch(efetch_inputs, efetch_config)

In [17]:
# Display output docs
display_api_reference("ncbi", "output", "run_ncbi_efetch")

# Inspect the fetched FASTA record
record = fetch_result.fasta_records[0]
print(f"Accession: {record.accession}")
print(f"Header: {record.header[:80]}...")
print(f"Sequence length: {len(record.sequence)}")
print(f"Preview: {record.sequence[:60]}...")

**Output** — `NCBIEfetchOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>fasta_records</code> | <code>list[NCBIFastaRecord]</code> | <code>[]</code> | Parsed FASTA records |
| <code>source_url</code> | <code>str</code> | required | Sanitized request URL |

**`NCBIFastaRecord`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>header</code> | <code>str</code> | required | FASTA header line |
| <code>sequence</code> | <code>str</code> | required | Sequence string |
| <code>accession</code> | <code>str &#124; None</code> | <code>None</code> | Accession extracted from header |

Accession: NP_000537.3
Header: NP_000537.3 cellular tumor antigen p53 isoform a [Homo sapiens]...
Sequence length: 393
Preview: MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP...


#### Genomic Subsequence Extraction

When working with large genomes, you often need just a small region rather than the entire chromosome. The efetch function supports coordinate-based extraction with strand selection, which is useful for pulling promoter regions, specific gene loci, or regulatory elements. Here we retrieve a 201 bp region from the lacI locus on the minus strand of the *E. coli* K-12 genome.

In [18]:
# Fetch a 201 bp region from the lacI locus (minus strand)
subseq_inputs = NCBIEfetchInput(
    db="nuccore",
    identifier="NC_000913.3",
    return_format="fasta",
    seq_start=366428,
    seq_stop=366628,
    strand="-",
)

subseq_result = run_ncbi_efetch(subseq_inputs, efetch_config)

record = subseq_result.fasta_records[0]
print(f"Accession: {record.accession}")
print(f"Sequence length: {len(record.sequence)}")
print(f"Preview: {record.sequence[:80]}...")
print(f"Source URL: {subseq_result.source_url}")

Accession: NC_000913.3:c366628-366428
Sequence length: 201
Preview: CTGCTGGGGCAAACCAGCGTGGACCGCTTGCTGCAACTCTCTCAGGGCCAGGCGGTGAAGGGCAATCAGCTGTTGCCCGT...
Source URL: https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id=NC_000913.3&rettype=fasta&retmode=text&seq_start=366428&seq_stop=366628&strand=2&tool=proto_tools_ncbi_fetch


### `run_ncbi_reference_bundle`

`run_ncbi_reference_bundle` downloads the NCBI Datasets bundle for a versioned assembly accession and returns paths to its genome FASTA, GFF3 annotation, transcript FASTA, and protein FASTA. By default the bundle is stored under `$PROTO_DATABASES_DIR/ncbi/<accession>/` and reused by later calls; here we extract the E. coli K-12 MG1655 reference (about 3 MB) into a temporary directory instead.

In [19]:
from tempfile import TemporaryDirectory

from proto_tools import (
    NCBIReferenceBundleConfig,
    NCBIReferenceBundleInput,
    run_ncbi_reference_bundle,
)

bundle_dir = TemporaryDirectory()

In [20]:
# Display docs
display_api_reference("ncbi", "input", "run_ncbi_reference_bundle")

# A versioned accession; an NCBI Datasets page link works too
bundle_inputs = NCBIReferenceBundleInput(accession="GCF_000005845.2", destination_dir=bundle_dir.name)

**Input** — `NCBIReferenceBundleInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>accession</code> | <code>str</code> | required | Versioned GCA_/GCF_ assembly accession, or an NCBI page link containing one |
| <code>destination_dir</code> | <code>str &#124; None</code> | <code>None</code> | Directory to extract into; defaults to the shared databases directory |

In [21]:
# Display config docs
display_api_reference("ncbi", "config", "run_ncbi_reference_bundle")

# Default config downloads all four annotation types | see docs above for all fields
bundle_config = NCBIReferenceBundleConfig()

**Config** — `NCBIReferenceBundleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>annotation_types</code> | <code>list[Literal['GENOME_FASTA', 'GENOME_GFF', 'RNA_FASTA', 'PROT_FASTA']]</code> | <code>['GENOME_FASTA', 'GENOME_GFF', 'RNA_FASTA', 'PROT_FASTA']</code> | NCBI Datasets reference files to download |
| <code>max_download_bytes</code> | <code>int</code> | <code>5000000000</code> | Largest compressed bundle to download; the human assembly is about 1.2 GB |
| <code>ncbi_api_key</code> | <code>str &#124; None</code> | <code>$NCBI_API_KEY</code> | Optional NCBI API key; raises the request rate limit. Falls back to $NCBI_API_KEY. |

In [22]:
# Download and extract the bundle
bundle = run_ncbi_reference_bundle(bundle_inputs, bundle_config)

In [23]:
from pathlib import Path

# Display output docs
display_api_reference("ncbi", "output", "run_ncbi_reference_bundle")

# Each path is None when the assembly has no such file
for field in ("genome_fasta", "genome_gff", "rna_fasta", "protein_fasta"):
    path = getattr(bundle, field)
    print(f"{field:14} {Path(path).name if path else None}")
print(f"\nGenome FASTA header: {Path(bundle.genome_fasta).open().readline().strip()}")

**Output** — `NCBIReferenceBundleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>accession</code> | <code>str</code> | required | Versioned assembly accession |
| <code>source_url</code> | <code>str</code> | required | NCBI Datasets download endpoint |
| <code>directory</code> | <code>str</code> | required | Directory holding the extracted bundle |
| <code>genome_fasta</code> | <code>str &#124; None</code> | <code>None</code> | Genomic nucleotide FASTA |
| <code>genome_gff</code> | <code>str &#124; None</code> | <code>None</code> | GFF3 genome annotation |
| <code>rna_fasta</code> | <code>str &#124; None</code> | <code>None</code> | Transcript nucleotide FASTA |
| <code>protein_fasta</code> | <code>str &#124; None</code> | <code>None</code> | Protein FASTA |

genome_fasta   GCF_000005845.2_ASM584v2_genomic.fna
genome_gff     genomic.gff
rna_fasta      None
protein_fasta  protein.faa

Genome FASTA header: >NC_000913.3 Escherichia coli str. K-12 substr. MG1655, complete genome


## Export Results

Results from all three functions can be exported to JSON format for downstream processing or archival. This is useful for building reproducible pipelines where intermediate results need to be saved between steps.

In [24]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

fetch_result.export("ncbi_efetch_results", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'ncbi_efetch_results.json'}")

summary_result.export("ncbi_gene_summary", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'ncbi_gene_summary.json'}")

Exported to example_output/ncbi_efetch_results.json
Exported to example_output/ncbi_gene_summary.json
